In [ ]:
"""
Quick Start Notebook - ALDF-EAM
==================================
10-minute introduction to Adaptive Leakage Detection Framework

This notebook demonstrates:
1. Loading synthetic AMI data
2. Computing S-vector
3. Running adaptive analysis
4. Interpreting results
"""



# ALDF-EAM Quick Start

**Goal:** Detect Non-Technical Losses (NTL) in partially-deployed AMI networks

**Dataset:** 60 consumers, 6 DTs, 60 days, 28% DT coverage (realistic India scenario)

**Time:** ~5 minutes to run all cells



## Step 1: Install Dependencies (First Time Only)



In [ ]:
# Uncomment if running locally (not needed in Colab)
# !pip install numpy pandas matplotlib seaborn scikit-learn scipy



## Step 2: Import Libraries



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")



## Step 3: Generate Synthetic AMI Data

**Scenario:**
- 60 residential consumers
- 6 Distribution Transformers (DTs)
- 60-day analysis window
- **Controlled anomalies:** 12 consumers with NTL (20% base rate)



In [ ]:
# Configuration
N_CONSUMERS = 60
N_DTS = 6
N_DAYS = 60
INTERVALS_PER_DAY = 48  # 30-minute intervals

# Consumer assignment to DTs
consumers_per_dt = N_CONSUMERS // N_DTS
dt_assignments = np.repeat(range(1, N_DTS+1), consumers_per_dt)

# Generate consumer metadata
consumer_data = pd.DataFrame({
    'consumer_id': [f'C_{i:03d}' for i in range(1, N_CONSUMERS+1)],
    'dt_id': [f'DT_{dt:02d}' for dt in dt_assignments],
    'sanctioned_load_kw': np.random.choice([3, 4, 5, 6], size=N_CONSUMERS, p=[0.3, 0.4, 0.2, 0.1]),
    'category': np.random.choice(['Residential', 'Commercial'], size=N_CONSUMERS, p=[0.85, 0.15]),
    'has_meter': np.random.choice([True, False], size=N_CONSUMERS, p=[0.75, 0.25])  # 75% coverage
})

# Mark DT metering status (28% coverage = 2 of 6 DTs unmetered)
dt_metering = {
    'DT_01': True, 'DT_02': True,
    'DT_03': False,  # Unmetered
    'DT_04': False,  # Unmetered  
    'DT_05': True, 'DT_06': True
}
consumer_data['dt_metered'] = consumer_data['dt_id'].map(dt_metering)

print(f"✅ Generated {N_CONSUMERS} consumers across {N_DTS} DTs")
print(f"   Consumer metering: {consumer_data['has_meter'].mean():.1%}")
print(f"   DT metering: {sum(dt_metering.values())/len(dt_metering):.1%}")



## Step 4: Generate Load Profiles

**Pattern:** Realistic residential consumption with daily/weekly cycles



In [ ]:
def generate_load_profile(consumer_id, n_days, is_theft=False, theft_type=None):
    """Generate 30-min interval load profile"""
    n_intervals = n_days * INTERVALS_PER_DAY
    timestamps = pd.date_range('2026-01-01', periods=n_intervals, freq='30min')
    
    # Base daily pattern (kW)
    hour = timestamps.hour + timestamps.minute/60
    daily_pattern = (
        0.3 +  # Baseline (refrigerator, standby)
        0.5 * np.sin((hour - 6) * np.pi / 12) * (hour >= 6) * (hour < 22) +  # Day activity
        1.2 * ((hour >= 18) * (hour < 22))  # Evening peak
    )
    
    # Add randomness
    noise = np.random.normal(0, 0.1, n_intervals)
    load = daily_pattern + noise
    load = np.maximum(load, 0.1)  # Min 0.1 kW
    
    # Apply theft patterns
    if is_theft:
        if theft_type == 'night_min':
            # Suppress nighttime load (bypass refrigerator)
            night_mask = (hour >= 0) & (hour < 6)
            load[night_mask] *= 0.3
        elif theft_type == 'step_change':
            # Sudden drop after day 30 (bypass installed)
            if n_intervals > 30 * INTERVALS_PER_DAY:
                load[30*INTERVALS_PER_DAY:] *= 0.45
        elif theft_type == 'coordinated':
            # Consistent 60% reduction (parallel bypass)
            load *= 0.40
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'kw_load': load,
        'kwh_interval': load * 0.5  # 30-min interval = 0.5 hour
    })

# Generate profiles for all consumers
print("Generating load profiles...")

# Mark NTL consumers (12 total, 20% rate)
ntl_consumers = {
    'C_023': 'night_min',
    'C_047': 'night_min',
    'C_015': 'night_min',
    'C_038': 'night_min',
    'C_007': 'step_change',
    'C_029': 'step_change',
    'C_051': 'step_change',
    'C_012': 'coordinated',
    'C_019': 'coordinated',
    'C_034': 'coordinated',
    'C_041': 'coordinated',
    'C_056': 'coordinated'
}

# Generate all profiles
interval_data_list = []
for idx, row in consumer_data.iterrows():
    cid = row['consumer_id']
    is_theft = cid in ntl_consumers
    theft_type = ntl_consumers.get(cid, None)
    
    profile = generate_load_profile(cid, N_DAYS, is_theft, theft_type)
    profile['consumer_id'] = cid
    profile['dt_id'] = row['dt_id']
    interval_data_list.append(profile)

interval_data = pd.concat(interval_data_list, ignore_index=True)

print(f"✅ Generated {len(interval_data):,} interval records")
print(f"   NTL consumers: {len(ntl_consumers)} ({len(ntl_consumers)/N_CONSUMERS:.1%})")



## Step 5: Compute S-Vector (System State)

**S = [O, D, T, B]**
- **O:** Observability (can we SEE what's happening?)
- **D:** Data Quality (can we TRUST what we see?)
- **T:** Technical Risk (is infrastructure stressed?)
- **B:** Behavioral Risk (do patterns look suspicious?)



In [ ]:
# O-Index: Observability
c_F = 1.0  # Feeder meter present (assume yes)
c_DT = sum(dt_metering.values()) / len(dt_metering)  # 4/6 = 0.67
c_C = consumer_data['has_meter'].mean()  # 0.75
c_TOP = 0.80  # Assume 80% topology verified
c_SYNC = 0.90  # Assume 90% time sync

O = 0.25*c_F + 0.20*c_DT + 0.20*c_C + 0.20*c_TOP + 0.15*c_SYNC
print(f"O (Observability): {O:.3f}")

# D-Index: Data Quality
d_compl = 0.94  # 94% data completeness (6% missing)
d_uptime = 0.92  # 92% communication uptime
d_tamper_avg = 0.85  # Average tamper indicator (low event rate)
d_health = 0.90  # 90% meters passing diagnostics

D = 0.30*d_compl + 0.25*d_uptime + 0.25*d_tamper_avg + 0.20*d_health
print(f"D (Data Quality): {D:.3f}")

# T-Index: Technical Risk
t_loading = 0.42  # 42% average loading (low risk)
T = t_loading
print(f"T (Technical Risk): {T:.3f}")

# B-Index: Behavioral Risk (computed from data)
# For now, use example value
B = 0.65
print(f"B (Behavioral Risk): {B:.3f}")

print("\n" + "="*50)
print(f"S-VECTOR: [{O:.3f}, {D:.3f}, {T:.3f}, {B:.3f}]")
print("="*50)

# Determine operating band
if D <= 0.40:
    band = "STAGE_A (Suspend output)"
elif D < 0.65:
    band = "TRANSITIONAL (Limited methods)"
else:
    band = "FULL_ANALYTICS (All methods)"

print(f"\nOperating Band: {band}")



## Step 6: Compute Behavioral Indicators

Six indicators that flag suspicious consumption patterns:



In [ ]:
# Compute behavioral indicators for each consumer
behavioral_scores = []

for cid in consumer_data['consumer_id'].unique():
    consumer_intervals = interval_data[interval_data['consumer_id'] == cid].copy()
    
    if len(consumer_intervals) == 0:
        continue
    
    # Extract hour for night minimum calculation
    consumer_intervals['hour'] = pd.to_datetime(consumer_intervals['timestamp']).dt.hour
    
    # b_NMR: Night Minimum Ratio
    night_load = consumer_intervals[consumer_intervals['hour'].between(2, 5)]['kw_load'].mean()
    avg_load = consumer_intervals['kw_load'].mean()
    nmr = night_load / avg_load if avg_load > 0 else 0
    
    if nmr < 0.12:
        b_nmr = 1.0
    elif nmr < 0.15:
        b_nmr = 0.7
    elif nmr < 0.18:
        b_nmr = 0.4
    else:
        b_nmr = 0.0
    
    # b_SCR: Step Change Ratio
    first_30_days = consumer_intervals[consumer_intervals['timestamp'] < '2026-01-31']['kw_load'].mean()
    last_30_days = consumer_intervals[consumer_intervals['timestamp'] >= '2026-01-31']['kw_load'].mean()
    
    if first_30_days > 0:
        scr = abs(last_30_days - first_30_days) / first_30_days
    else:
        scr = 0
    
    if scr > 0.60:
        b_scr = 1.0
    elif scr > 0.40:
        b_scr = 0.8
    elif scr > 0.25:
        b_scr = 0.5
    else:
        b_scr = 0.0
    
    # b_peer_dev: Simplified peer comparison
    # (Full implementation would group by sanctioned_load, category, etc.)
    peer_avg = interval_data.groupby('consumer_id')['kw_load'].mean().median()
    Z_score = abs(avg_load - peer_avg) / (peer_avg * 0.15) if peer_avg > 0 else 0
    b_peer_dev = min(1.0, Z_score / 5.0)  # Normalize to [0, 1]
    
    # Simplified B-index (normally weighted average of 6 indicators)
    B_consumer = 0.25*b_nmr + 0.25*b_scr + 0.20*b_peer_dev
    
    behavioral_scores.append({
        'consumer_id': cid,
        'b_NMR': b_nmr,
        'b_SCR': b_scr,
        'b_peer_dev': b_peer_dev,
        'B_index': B_consumer,
        'avg_daily_kwh': avg_load * 24,
        'night_min_kw': night_load,
        'nmr': nmr,
        'scr': scr
    })

behavioral_df = pd.DataFrame(behavioral_scores)

# Merge with ground truth
behavioral_df['is_ntl'] = behavioral_df['consumer_id'].isin(ntl_consumers.keys())

print(f"✅ Computed behavioral scores for {len(behavioral_df)} consumers")

# Show top suspects
top_suspects = behavioral_df.sort_values('B_index', ascending=False).head(10)
print("\nTop 10 Suspects by B-index:")
print(top_suspects[['consumer_id', 'B_index', 'b_NMR', 'b_SCR', 'is_ntl']].to_string(index=False))



## Step 7: Visualize Results



In [ ]:
# Plot 1: S-Vector Radar Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Radar chart
categories = ['O\n(Observability)', 'D\n(Data Quality)', 'T\n(Tech Risk)', 'B\n(Behavioral)']
values = [O, D, T, B]

angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
values += values[:1]  # Complete the circle
angles += angles[:1]

ax = plt.subplot(121, projection='polar')
ax.plot(angles, values, 'o-', linewidth=2, color='#028090', markersize=8)
ax.fill(angles, values, alpha=0.25, color='#02C39A')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75, 1.0])
ax.grid(True)
ax.set_title('S-Vector: System State\n', fontsize=13, fontweight='bold', pad=20)

# Plot 2: B-index distribution
ax2 = axes[1]
ax2.hist(behavioral_df[~behavioral_df['is_ntl']]['B_index'], bins=15, alpha=0.6, label='Legitimate', color='green')
ax2.hist(behavioral_df[behavioral_df['is_ntl']]['B_index'], bins=15, alpha=0.6, label='NTL (ground truth)', color='red')
ax2.axvline(0.60, color='orange', linestyle='--', linewidth=2, label='B_high threshold (0.60)')
ax2.set_xlabel('B-index (Behavioral Risk)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Behavioral Risk Distribution', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



## Step 8: Performance Metrics



In [ ]:
# Generate risk scores (simplified: just use B-index)
behavioral_df['risk_score'] = behavioral_df['B_index']

# Rank consumers
behavioral_df['rank'] = behavioral_df['risk_score'].rank(ascending=False, method='first').astype(int)

# Calculate metrics at top-25 cutoff
top_25 = behavioral_df[behavioral_df['rank'] <= 25]

true_positives = top_25['is_ntl'].sum()
false_positives = (~top_25['is_ntl']).sum()
false_negatives = behavioral_df[(behavioral_df['rank'] > 25) & (behavioral_df['is_ntl'])].shape[0]

IHR = true_positives / len(ntl_consumers)  # Inspection Hit Rate
FPR = false_positives / 25  # False Positive Rate (of top 25)
precision = true_positives / 25
recall = IHR
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("="*60)
print("PERFORMANCE METRICS (Top 25 Suspects)")
print("="*60)
print(f"Inspection Hit Rate (IHR):  {IHR:.1%}  (Target: ≥60%)")
print(f"False Positive Rate (FPR):  {FPR:.1%}  (Target: ≤30%)")
print(f"Precision:                  {precision:.1%}")
print(f"Recall:                     {recall:.1%}")
print(f"F1 Score:                   {f1:.2f}  (Target: ≥0.65)")
print("="*60)

# Show which NTL consumers were caught
caught = behavioral_df[(behavioral_df['rank'] <= 25) & (behavioral_df['is_ntl'])]['consumer_id'].tolist()
missed = behavioral_df[(behavioral_df['rank'] > 25) & (behavioral_df['is_ntl'])]['consumer_id'].tolist()

print(f"\n✅ Caught in top 25: {caught}")
print(f"❌ Missed (rank >25): {missed}")



## Step 9: Field Investigation Brief

**For utility field inspectors:**



In [ ]:
# Generate field inspection brief for top 5 suspects
print("\n" + "="*60)
print("FIELD INVESTIGATION BRIEF - TOP 5 SUSPECTS")
print("="*60)

for rank in range(1, 6):
    suspect = behavioral_df[behavioral_df['rank'] == rank].iloc[0]
    
    print(f"\n{'='*60}")
    print(f"RANK #{rank}: {suspect['consumer_id']}")
    print(f"{'='*60}")
    print(f"Risk Score:       {suspect['risk_score']:.2f}")
    print(f"Confidence:       {'HIGH' if suspect['B_index'] > 0.70 else 'MEDIUM'}")
    print(f"")
    print(f"Indicators:")
    print(f"  • Night Min Ratio: {suspect['nmr']:.3f} (normal: 0.20-0.30)")
    if suspect['b_NMR'] > 0.7:
        print(f"    → SUSPICIOUS: Nighttime load too low ({suspect['night_min_kw']:.2f} kW)")
    print(f"  • Step Change: {suspect['scr']:.1%}")
    if suspect['b_SCR'] > 0.5:
        print(f"    → SUSPICIOUS: Sudden consumption drop detected")
    print(f"  • Peer Deviation: b_peer_dev={suspect['b_peer_dev']:.2f}")
    if suspect['b_peer_dev'] > 0.50:
        print(f"    → SUSPICIOUS: Significantly below similar consumers")
    print(f"")
    print(f"Recommended Action:")
    if suspect['is_ntl']:
        print(f"  ✅ CONFIRMED NTL (ground truth) - Expect to find bypass/tamper")
    else:
        print(f"  ⚠️  Investigate - Could be false positive")
    print(f"")
    print(f"Location: {suspect['consumer_id'].replace('C_', 'Address_')}")



## Step 10: Next Steps

**You've completed the quick start! 🎉**

### What you learned:
- ✅ How ALDF-EAM computes the S-vector
- ✅ Six behavioral indicators (NMR, SCR, peer deviation, etc.)
- ✅ Three-band operating model (Stage A / Transitional / Full)
- ✅ Performance metrics (IHR, FPR, F1)

### Explore further:
1. **Full Demo Notebook:** [`ALDF_EAM_Implementation.ipynb`](ALDF_EAM_Implementation.ipynb)
   - Complete workflow with all 6 methods (M1-M6)
   - Bayesian fusion, feedback loop
   - Real India AMI data analysis

2. **MDM Integration:** [`docs/integration/mdm_integration.md`](docs/integration/mdm_integration.md)
   - Connect to Oracle, Itron, Landis+Gyr MDM
   - Production deployment examples

3. **Method Deep Dives:**
   - S-Vector calculation: [`notebooks/S_Vector_Calculation.ipynb`](notebooks/S_Vector_Calculation.ipynb)
   - Method comparison: [`notebooks/Method_Comparison.ipynb`](notebooks/Method_Comparison.ipynb)

4. **Deploy ALDF-EAM:**
   ```bash
   pip install aldf-eam
   python examples/basic_usage.py
   ```

### Questions?
- GitHub Discussions: https://github.com/ap1singh/ALDF-EAM/discussions
- Documentation: [`docs/getting_started.md`](docs/getting_started.md)
- Contact: [your.email@domain.com]

**Star the repo if you found this useful!** ⭐
https://github.com/ap1singh/ALDF-EAM

print("\n" + "="*60)
print("✅ QUICK START COMPLETE!")
print("="*60)
print("\nNext: Run the full demo notebook for complete workflow")
print("→ ALDF_EAM_Implementation.ipynb")

